# AoC 2024 Day 11 — Plutonian Pebbles

**Spark — explode + groupBy counting**

Puzzle: <https://adventofcode.com/2024/day/11>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A line of stones, each engraved with a number. Every blink, all stones change at once, each by the **first** rule that applies:

1. a `0` becomes a `1`
2. a number with an **even** digit count splits into two stones — the left half of the digits and the right half, with leading zeroes dropped (`1000` → `10` and `0`)
3. otherwise the number is multiplied by **2024**

- **Part 1** — blink 25 times and count how many stones you end up with.

## The approach

The whole day turns on one observation: **the puzzle asks how many stones, not which stones in what order.**

The rules are per-stone and depend on nothing but the number engraved on it. A `5` behaves identically whether it sits at the front of the line or the back, and the line never interacts with itself. So the order the puzzle so carefully preserves is decoration — it changes the *arrangement*, never the *count*.

That lets the line be stored as a frequency table, `(stone, count)`, instead of a sequence. One blink becomes:

- **explode** — each stone maps to an array of 1 or 2 children (`when` on the three rules)
- **groupBy + sum** — children that collide merge, carrying their counts

The payoff is the size difference. After 25 blinks the example line holds 55312 stones but only a few thousand *distinct* numbers, and that gap widens every blink — the split rule keeps producing small numbers that everything else eventually falls into. Simulating the line grows exponentially; the frequency table plateaus.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day11

spark = get_spark('aoc-2024-day11')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '125 17\n'

print('part 1:', day11.part1(spark, EXAMPLE), '(expected 55312)')

### Distinct rows vs. stones

Two numbers per blink. The right one is what the puzzle counts; the left one is what Spark actually stores.

In [ ]:
from pyspark.sql import functions as F

rows = [(int(v), 1) for v in EXAMPLE.split()]
stones = spark.createDataFrame(rows, 'stone LONG, count LONG').groupBy('stone').agg(
    F.sum('count').alias('count')
)

# Distinct rows vs. stones in the line -- the gap is the whole optimisation.
for blink in range(7):
    total = stones.agg(F.sum('count').alias('t')).collect()[0]['t']
    print(f'blink {blink:>2}: {stones.count():>4} distinct  |  {total:>6} stones in the line')
    stones = day11._blink(stones).localCheckpoint()

stones.orderBy('stone').show(10)

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 11)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day11.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day11 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- **`localCheckpoint()` per blink is load-bearing.** Without it the 25 blinks stack into a single unexecuted plan 25 group-bys deep, and the analyzer alone starts to crawl before any data moves. The checkpoint truncates the lineage after each round, which is the standard shape for any iterative Spark job.
- Splitting is done on the **string** form: `length(text) % 2 == 0`, then two `substring` slices cast back to `long`. `substring` is **1-based**, so the halves are `(1, half)` and `(half + 1, half)`.
- The cast back to `long` is what strips leading zeroes for free — `1000` splits to `10` and `00`, and `00` becomes `0`, exactly as the rules require. Under ANSI mode this cast is still safe because every substring is pure digits.
- Stones are `LONG`, not `INT`. Multiplying by 2024 repeatedly overflows 32 bits quickly, and Spark would silently wrap (or, under ANSI mode, raise) rather than widen.
- The input is whitespace-separated on **one line**, so `data.split()` with no argument is the parse — no newline handling needed.
- Duplicate stones in the initial line must be folded with a `groupBy` *before* the first blink; otherwise the same number appears as two rows and the counts, while not wrong, stop being canonical.